# FP3.4

## Group Members
- Nolan Cummins (me)

## About
For my final project, I decided to look at the relationship between the ruling party at any given time and the cost of very valuable important goods, like spaghetti. My first inclination was that Republicans would obviously run the economy into the ground, and although that may appear so when watching the stock market, the actual price of goods may appear volatile for a year or two, but after adjusting for inflation, eventually it stabilizes. Since large economical decisions typically take time to propogate into significant impacts on commodity pricing, when switching from one administration to another of the opposite party, it becomes difficult to determine if the actions of one truly made everything more expensive. To investigate this, I made two visualizations: a line chart and a geomap. 

### 2nd Visualization:
Fundamentally, the government consists of 3 branches: executive, judicial, and legislative. To determine an objective "ruling party," I took the weighted sum for each branch (including House/Senate); whichever holds >51% is considering "in-charge." This is shown as the shaded background (red/blue) in the first visualization. There was a lot of ``Pandas`` data cleaning to do, but I also included the inflation-adjusted prices of various commodities and goods as a line-chart. The bar chart on the right shows the average distribution during the selected years of the government by party for each branch (and House/Senate). This gives a good idea of the continuous changes in prices over-time.

### 1st Visualization:
However, considering change happens from the top, and takes time to propogate down, I wanted to see if the more localized events had any impact. There are two ways to enforce laws: violence, and in the courts. I'm going to ignore violence for now, as that's hard to quantify. We can take a look at the distribution of Republican-appointed and Democrat-appointed judges for each state by year to get an idea of which side of the 'court,' the courts will side with. A more right-leaning judicial system would be more likely to strongly enforce Republican policies, strike down Democrat policies, and issue rulings that most align with the Republican party, for instance. These decisions could have small impacts on companies, people, and subsequently how things are priced and demanded by the public. If we zoom out and look at the sum, those small decisions have the potential for wide-sweeping effects downstream. For this, I made a geoplot with the 50 states shaded according to their judicial makeup, and the plot on the right shows the average commodity price(s) for that year.

## Data Sources:
1. Visualization 1: 
    - Commodity Data: [Bureau of Labor Statistics (LABSTAT)](https://www.bls.gov/data/)
    - Political Data: [United States Project (GitHub)](https://github.com/unitedstates/congress-legislators)
2. Visualization 2:
    - Judicial Data: [Free Law Project](https://wiki.free.law/c/courtlistener/help/api/bulk-data/bulk-legal-data)

## Visualization 1 Data

Starting with the cell below, this serves as the primary data processing engine for the project, responsible for cleaning, normalizing, and synchronizing disparate datasets from the Bureau of Labor Statistics and historical government records.

* **Inflation Adjustment:** It calculates a baseline inflation scale using the Consumer Price Index (CPI) to convert historical "nominal" prices into "real" current-day dollars, allowing for a fair comparison of commodity costs over 50 years.
* **Data Normalization:** It extracts and cleans complex JSON and CSV records for the Executive, Legislative (House and Senate), and Judicial branches, standardizing date formats and party affiliations.
* **Political Power Calculation:** To determine which party is "in charge" at any given month, the code calculates a weighted average of power across all three branches. It treats the government as a balanced system where the Legislative, Executive, and Judicial branches each hold a one-third share of influence.
* **Time-Series Alignment:** It builds a master monthly timeline from 1976 to the present, ensuring that commodity price fluctuations are perfectly aligned with the exact political makeup of the government at that specific moment.

By consolidating these variables into a single synchronized dataframe (`dash_df`), we can objectively observe how shifts in federal and judicial leadership correlate with the real-world cost of living.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ap_url = r"data/ap.series.txt"
gas_url = r"data/ap.data.2.Gasoline.txt"
food_url = r"data/ap.data.3.Food.txt"
cu_url = r"data/cu.series.txt"
cpi_url = r"data/cu.data.1.AllItems.txt"
current_legislators_url = r"data/legislators-current.json"
historical_legislators_url = r"data/legislators-historical.json"
executive_url = r"data/executive.json"
judicial_url = r"data/justicesdata2022.csv"

print("Downloading datasets...")
# handle specialized spacing in bls text files using regex separator
ap_series = pd.read_csv(ap_url, sep=r'\s*\t\s*', engine='python', index_col=0) 
gasoline_series = pd.read_csv(gas_url, sep=r'\s*\t\s*', engine='python', index_col=0)
food_series = pd.read_csv(food_url, sep=r'\s*\t\s*', engine='python', index_col=0) # Added food data load
cu_series = pd.read_csv(cu_url, sep=r'\s*\t\s*', engine='python', index_col=0)
cpi_series = pd.read_csv(cpi_url, sep=r'\s*\t\s*', engine='python', index_col=0)
current_legislators = pd.read_json(current_legislators_url) 
historical_legislators = pd.read_json(historical_legislators_url) 
executive = pd.read_json(executive_url)
judicial = pd.read_csv(judicial_url)
print("Done!")

# Combine raw commodity data for easier lookup in the function
all_commodities_series = pd.concat([gasoline_series, food_series])

print("Calculating baseline inflation...")
cpi_id = "CUSR0000SA0"
# select specific consumer price index series for all urban consumers
cpi_data = cpi_series.loc[cpi_id].copy()
cpi_data["value"] = pd.to_numeric(cpi_data["value"], errors='coerce')
# convert bls year and month period codes into standard datetime objects
cpi_data["date"] = pd.to_datetime(cpi_data["year"].astype(str) + cpi_data["period"].str.replace("M", ""), format="%Y%m")
# calculate year over year percentage change for inflation tracking
cpi_data['inflation_rate'] = cpi_data['value'].pct_change(periods=12, fill_method=None) * 100
current_cpi_val = cpi_data['value'].iloc[-1]

def get_adjusted_commodity(series_id, raw_data, metadata, cpi_df, current_cpi):
    """Cleans commodity data and adjusts prices for inflation."""
    title = metadata.loc[series_id]["series_title"]
    print(f"Processing & Adjusting: '{title}'")
    
    # Extract & clean
    df = raw_data.loc[series_id].copy()
    if isinstance(df, pd.Series): # Handle single-row returns gracefully
        df = df.to_frame().T
        
    df["value"] = pd.to_numeric(df["value"], errors='coerce')
    df["date"] = pd.to_datetime(df["year"].astype(str) + df["period"].str.replace("M", ""), format="%Y%m")
    
    # Merge with CPI and calculate real price
    # align monthly commodity prices with corresponding monthly cpi values
    merged = pd.merge(df, cpi_df[['date', 'value']], on='date', how='inner')
    merged.rename(columns={'value_x': 'price', 'value_y': 'cpi'}, inplace=True)
    # adjust historical price to current dollar value using cpi ratio
    merged['real_price'] = merged['price'] * (current_cpi / merged['cpi'])
    
    return merged

item_ids = {
    "gas": "APU000074714",
    "spaghetti": "APU0000701321",
    "eggs": "APU0000708111",
    "icecream": "APU0000710411",
    "beans": "APU0000714233",
    "bacon": "APU0000704111",
    "gallonmilk": "APU0000709112"
}

# Dictionary to hold all the cleaned, inflation-adjusted dataframes
adjusted_dataframes = {}

for name, s_id in item_ids.items():
    adjusted_dataframes[name] = get_adjusted_commodity(
        series_id=s_id, 
        raw_data=all_commodities_series, 
        metadata=ap_series, 
        cpi_df=cpi_data, 
        current_cpi=current_cpi_val
    )


print("Organizing presidents...")
exec_records = []
# flatten nested presidential term data from json into individual records
for _, row in executive.iterrows():
    for term in row['terms']:
        if term['type'] == 'prez':
            exec_records.append({'start': term['start'], 'end': term.get('end'), 'party': term.get('party')})
df_pres = pd.DataFrame(exec_records)
df_pres['start'] = pd.to_datetime(df_pres['start'])
# assume current office holders are active until today for time series overlap
df_pres['end'] = pd.to_datetime(df_pres['end'], errors='coerce').fillna(pd.Timestamp.today())

print("Extracting justices...")
# filter for supreme court justices using serve code prefix
df_jud = judicial[judicial['serve'].astype(str).str.startswith('1')].copy()
# replace placeholder strings for active justices with current timestamp
df_jud['datesere'] = df_jud['datesere'].astype(str).replace(
    '999. JUSTICE STILL ON COURT', 
    pd.Timestamp.today().strftime('%m/%d/%Y')
)
df_jud['term_start'] = pd.to_datetime(df_jud['dateserb'], errors='coerce')
df_jud['term_end'] = pd.to_datetime(df_jud['datesere'], errors='coerce').fillna(pd.Timestamp.today())
party_mapping = {'1. democrat': 'Democrat', '6. republican': 'Republican'}
# map numeric party codes to descriptive labels
df_jud['party'] = df_jud['parnom'].astype(str).str.strip().str.lower().map(party_mapping)

print("Merging legislator datasets and extracting parties...")
rep_records = []
sen_records = []
combined_legislators = pd.concat([current_legislators, historical_legislators], ignore_index=True)

# iterate through nested json to separate house and senate member terms
for _, row in combined_legislators.iterrows():
    if isinstance(row.get('terms'), list):
        for term in row['terms']:
            if term.get('type') == 'rep':
                rep_records.append({'start': term['start'], 'end': term['end'], 'party': term.get('party'), 'state': term.get('state')})
            elif term.get('type') == 'sen':
                sen_records.append({'start': term['start'], 'end': term['end'], 'party': term.get('party'), 'state': term.get('state')})

df_rep = pd.DataFrame(rep_records)
df_rep['start'] = pd.to_datetime(df_rep['start'], errors='coerce')
df_rep['end'] = pd.to_datetime(df_rep['end'], errors='coerce').fillna(pd.Timestamp.today())

df_sen = pd.DataFrame(sen_records)
df_sen['start'] = pd.to_datetime(df_sen['start'], errors='coerce')
df_sen['end'] = pd.to_datetime(df_sen['end'], errors='coerce').fillna(pd.Timestamp.today())

def get_party_counts(df, date, start_col='start', end_col='end'):
    # find all individuals holding office at a specific point in time
    active = df[(df[start_col] <= date) & (df[end_col] > date)]
    return active['party'].value_counts().to_dict()

print("Building Time Series...")
# create monthly intervals for longitudinal analysis
dates = pd.date_range(start="1976-01-01", end=pd.Timestamp.today(), freq="MS")
rep_ts, sen_ts, jud_ts, pres_ts = [], [], [], []

for d in dates:    
    rep_counts = get_party_counts(df_rep, d)
    rep_ts.append({'Date': d, 'Democrat': rep_counts.get('Democrat', 0), 'Republican': rep_counts.get('Republican', 0)})
    
    sen_counts = get_party_counts(df_sen, d)
    sen_ts.append({'Date': d, 'Democrat': sen_counts.get('Democrat', 0), 'Republican': sen_counts.get('Republican', 0)})

    j_counts = get_party_counts(df_jud, d, 'term_start', 'term_end')
    jud_ts.append({'Date': d, 'Democrat': j_counts.get('Democrat', 0), 'Republican': j_counts.get('Republican', 0)})
    
    active_pres = df_pres[(df_pres['start'] <= d) & (df_pres['end'] > d)]
    pres_party = active_pres['party'].iloc[0] if not active_pres.empty else None
    pres_ts.append({'Date': d, 'Party': pres_party})

print("Converting to dataframes...")
rep_df = pd.DataFrame(rep_ts).set_index('Date')
sen_df = pd.DataFrame(sen_ts).set_index('Date')
judicial_df = pd.DataFrame(jud_ts).set_index('Date')
executive_df = pd.DataFrame(pres_ts).set_index('Date')
print("Done!")

# calculate percentage share of power for each party per branch
rep_total = rep_df['Democrat'] + rep_df['Republican']
sen_total = sen_df['Democrat'] + sen_df['Republican']
jud_total = judicial_df['Democrat'] + judicial_df['Republican']

dem_rep_weight = (rep_df['Democrat'] / rep_total)
rep_rep_weight = (rep_df['Republican'] / rep_total)

dem_sen_weight = (sen_df['Democrat'] / sen_total)
rep_sen_weight = (sen_df['Republican'] / sen_total)

dem_leg_weight = (dem_rep_weight + dem_sen_weight)
rep_leg_weight = (rep_rep_weight + rep_sen_weight)

dem_jud_weight = (judicial_df['Democrat'] / jud_total)
rep_jud_weight = (judicial_df['Republican'] / jud_total)

dem_exec_weight = (executive_df['Party'] == 'Democrat').astype(float)
rep_exec_weight = (executive_df['Party'] == 'Republican').astype(float)

dash_df = pd.DataFrame({
    'House_Dem': dem_rep_weight,       'House_Rep': rep_rep_weight,
    'Sen_Dem': dem_sen_weight,         'Sen_Rep': rep_sen_weight,
    'Jud_Dem': dem_jud_weight,         'Jud_Rep': rep_jud_weight,
    'Exec_Dem': dem_exec_weight,       'Exec_Rep': rep_exec_weight
}, index=rep_df.index)

# weight branches equally giving legislative and judicial and executive one third share each
dem_total_scaled = ((dash_df['House_Dem'] + dash_df['Sen_Dem']) / 2 / 3) + (dash_df['Jud_Dem'] / 3) + (dash_df['Exec_Dem'] / 3)
rep_total_scaled = ((dash_df['House_Rep'] + dash_df['Sen_Rep']) / 2 / 3) + (dash_df['Jud_Rep'] / 3) + (dash_df['Exec_Rep'] / 3)
# determine dominant party based on majority threshold across all government branches
dash_df['Party in Charge'] = np.where(dem_total_scaled > 0.5, 'Democrat', np.where(rep_total_scaled > 0.5, 'Republican', None))

Done!
Calculating baseline inflation...
Processing & Adjusting: 'Gasoline, unleaded regular, per gallon/3.785 liters in U.S. city average, average price, not seasonally adjusted'
Processing & Adjusting: 'Spaghetti (cost per pound/453.6 grams) in U.S. city average, average price, not seasonally adjusted'
Processing & Adjusting: 'Eggs, grade A, large, per doz. in U.S. city average, average price, not seasonally adjusted'
Processing & Adjusting: 'Ice cream, prepackaged, bulk, regular, per 1/2 gal. (1.9 lit) in U.S. city average, average price, not seasonally adjusted'
Processing & Adjusting: 'Beans, dried, any type, all sizes, per lb. (453.6 gm) in U.S. city average, average price, not seasonally adjusted'
Processing & Adjusting: 'Bacon, sliced, per lb. (453.6 gm) in U.S. city average, average price, not seasonally adjusted'
Processing & Adjusting: 'Milk, fresh, whole, fortified, per gal. (3.8 lit) in U.S. city average, average price, not seasonally adjusted'
Organizing presidents...
Extr

This output tracks the step-by-step execution of a data pipeline that downloads raw economic and political datasets, adjusts commodity prices for inflation, and synchronizes government leadership records into a unified time-series format.

In [2]:
dash_df.head()

,House_Dem,House_Rep,Sen_Dem,Sen_Rep,Jud_Dem,Jud_Rep,Exec_Dem,Exec_Rep,Party in Charge
Date,,,,,,,,,
1976-01-01,0.671910,0.328090,0.622449,0.377551,0.444444,0.555556,0.0,1.0,Republican
1976-02-01,0.671171,0.328829,0.622449,0.377551,0.444444,0.555556,0.0,1.0,Republican
1976-03-01,0.671171,0.328829,0.622449,0.377551,0.444444,0.555556,0.0,1.0,Republican
1976-04-01,0.671171,0.328829,0.622449,0.377551,0.444444,0.555556,0.0,1.0,Republican
1976-05-01,0.669663,0.330337,0.622449,0.377551,0.444444,0.555556,0.0,1.0,Republican


# Visualization 1 Chart

This cell below transforms the processed time-series data into an interactive dashboard that correlates political leadership with economic trends. It utilizes `Altair` to create a multi-layered, synchronized visualization.

* **Data Reshaping for Visualization:** The code converts monthly commodity prices and government makeup statistics into "long-format" DataFrames. This is a critical step for Altair, as it allows the charting engine to dynamically group data by commodity name or government branch.
* **Temporal Block Generation:** It calculates "blocks" of political time by identifying consecutive months where the same party was in power. These blocks are used to create the background color-shading (red for Republican, blue for Democrat) that sits behind the price lines.
* **Interactive Brushing:** A `brush` selection is implemented on the main line chart. This allows users to click and drag to select a specific window of time (e.g., a specific presidency or economic cycle).
* **Dynamic Cross-Filtering:** The bar chart on the right is linked to the line chart via the brush selection. When a user selects a time range, the bar chart automatically updates to show the average distribution of power across the Executive, Judicial, and Legislative branches for only that selected period.
* **Visual Layering:** The final output uses a layered approach, stacking translucent party-colored rectangles behind darkened line shadows and bright data lines to ensure maximum legibility against complex backgrounds.

The result is a synchronized dashboard (`visualization1.json`) that allows for a direct visual comparison between party control and the fluctuating cost of essential goods.

In [9]:
import altair as alt

# bypass default limits to allow plotting large datasets
alt.data_transformers.disable_max_rows()

price_list = []
# collect monthly prices from individual tables into a single list
for name, df in adjusted_dataframes.items():
    if not df.empty and 'date' in df.columns and 'real_price' in df.columns:
        temp = df[['date', 'real_price']].copy()
        temp['commodity'] = name
        price_list.append(temp)

# combine all commodities into one long table for charting
df_prices = pd.concat(price_list)
# strip timezone info to prevent visualization errors
df_prices['date'] = pd.to_datetime(df_prices['date']).dt.tz_localize(None)
df_prices = df_prices.dropna(subset=['real_price'])
df_prices = df_prices.sort_values(by=['commodity', 'date'])

records = []
current_party = None
start_date = None

# prepare data for background shading based on party control
temp_dash = dash_df[['Party in Charge']].copy()
temp_dash.index = pd.to_datetime(temp_dash.index).tz_localize(None)

# group consecutive months of the same party into blocks with start and end dates
for date, row in temp_dash.iterrows():
    party = row['Party in Charge']
    if party != current_party:
        if current_party is not None:
            records.append({'start': start_date, 'end': date, 'Party': current_party})
        current_party = party
        start_date = date
        
if current_party is not None:
    records.append({'start': start_date, 'end': temp_dash.index.max(), 'Party': current_party})

df_blocks = pd.DataFrame(records).dropna(subset=['Party'])

df_makeup = dash_df.drop(columns=['Party in Charge']).copy()
df_makeup.index = pd.to_datetime(df_makeup.index).tz_localize(None)
# remove duplicate columns and set up date as a standard column
df_makeup = df_makeup.reset_index().rename(columns={'index': 'date', 'Date': 'date'})
df_makeup = df_makeup.loc[:, ~df_makeup.columns.duplicated()]

# flatten branch data into long format to allow dynamic grouping in bar chart
df_makeup = df_makeup.melt(id_vars='date', var_name='Metric', value_name='Share')
# extract branch name and party name from the metric labels
df_makeup[['Branch', 'Party']] = df_makeup['Metric'].str.split('_', expand=True)
df_makeup['Branch'] = df_makeup['Branch'].replace({'Sen': 'Senate', 'Jud': 'Judicial', 'Exec': 'Executive'})
df_makeup['Party'] = df_makeup['Party'].replace({'Dem': 'Democrat', 'Rep': 'Republican'})


party_colors = alt.Scale(domain=['Democrat', 'Republican'], range=['blue', 'red'])
# define interactive area that allows users to select time ranges by dragging
brush = alt.selection_interval(encodings=['x'])

# Rename for alignment
df_blocks_aligned = df_blocks.rename(columns={'start': 'date', 'end': 'date_end'})

# Ensure they share the exact same temporal type
df_blocks_aligned['date'] = pd.to_datetime(df_blocks_aligned['date'])
df_prices['date'] = pd.to_datetime(df_prices['date'])


lines = alt.Chart(df_prices).mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('real_price:Q', title='Price ($)', scale=alt.Scale(zero=False)),
    color=alt.Color('commodity:N', scale=alt.Scale(scheme='dark2')),
    tooltip=['date:T', 'commodity:N', 'real_price:Q']
).add_params(brush)

shading_ribbon = alt.Chart(df_blocks_aligned).mark_rect().encode(
    x=alt.X('date:T', title=None, axis=alt.Axis(labels=True, ticks=False)),
    x2='date_end:T',
    color=alt.Color('Party:N', scale=party_colors, legend=None)
).properties(
    width=800, 
    height=20
)

bar_chart = alt.Chart(df_makeup).transform_calculate(
    date="toDate(datum.date)"
).transform_filter(
    # filter the bar chart based on the time selection made on the line chart
    brush
).mark_bar().encode(
    x=alt.X('Branch:N', sort=['House', 'Senate', 'Judicial', 'Executive'], title='Branch / House'),
    y=alt.Y('mean(Share):Q', scale=alt.Scale(domain=[0, 1]), title='Average Share of Power', axis=alt.Axis(format='%')),
    color=alt.Color('Party:N', scale=party_colors),
    tooltip=['Branch', 'Party', alt.Tooltip('mean(Share):Q', format='.1%')]
).properties(
    width=300, height=350, title="Avg. Government Makeup (Selected Range)"
)


brush = alt.selection_interval(encodings=['x'])

# create vertical blocks of color to represent periods of party control
background = alt.Chart(df_blocks_aligned).mark_rect(opacity=0.2).encode(
    x=alt.X('date:T'),
    x2=alt.X2('date_end'),          # alt.X2(), not shorthand string
    color=alt.Color('Party:N', scale=party_colors, legend=None)
)

lines_only = alt.Chart(df_prices).mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('real_price:Q', title='Price ($)', scale=alt.Scale(zero=False)),
    color=alt.Color('commodity:N', scale=alt.Scale(scheme='tableau10')),
    tooltip=['date:T', 'commodity:N', 'real_price:Q']
).add_params(brush)

# create a slightly offset black line to improve legibility against background colors
shadow = alt.Chart(df_prices).mark_line(
    strokeWidth=4, 
    strokeOpacity=0.3, 
    color='black'
).encode(
    x=alt.X('date:T'),
    y=alt.Y('real_price:Q'),
    detail='commodity:N'
)

# stack background and data layers and allow independent color scales
left_panel = alt.layer(background, shadow, lines_only).resolve_scale(
    color='independent'
).properties(width=800, height=350, title="Avg. Year-by-Year Commodity Prices by Ruling Party")

# join charts horizontally and keep color schemes separate
dashboard = (left_panel | bar_chart).resolve_scale(color='independent')
dashboard.save('visualization1.json')
dashboard.display()

alt.HConcatChart(...)

# Visualization 2 Data

This cell handles the ingestion and spatial-temporal mapping of the federal judiciary, focusing on identifying the political "lean" of judges across all 50 states over the last 50 years.

* **Presidential Timeline Normalization:** It processes historical executive records to create a continuous monthly lookup table of presidential party affiliations. This allows the script to determine which party was responsible for a specific judge's appointment based on their start date.
* **Judicial Record Cleaning:** The code sanitizes the "people-db" dataset, handling irregular date formats (e.g., converting "00" day placeholders to "01") and ensuring that active judges are correctly identified even if they have not yet reached a termination date.
* **Article III Filtering:** To focus on political influence, the script filters for Federal District (`FD`) and Appellate (`F`) courts. These represent the primary presidential appointments that shape legal policy within individual states.
* **Heuristic State Mapping:** Since judicial records often lack clean state codes, the code uses a tiered priority system. It first checks for explicit location data, and if missing, performs a string-search within the full court name (e.g., "District Court, D. Montana") to assign the judge to the correct state.
* **Yearly Snapshot Aggregation:** The script iterates through every year from 1976 to 2026, taking a "snapshot" of the bench on January 1st. It de-duplicates individual judges holding multiple roles to ensure an accurate count of Republican vs. Democrat appointees per state.

The resulting `final_df` provides the foundational data for the geographic visualization, mapping the judicial evolution of the United States through the lens of appointing authority.

In [4]:
positions = pd.read_csv('data/people-db-positions-2026-03-31.csv', low_memory=False, on_bad_lines='skip')
courts = pd.read_csv('data/courts-2026-03-31.csv', on_bad_lines='skip')
executive = pd.read_json('data/executive.json')

# Normalize Presidential Data (Democratic -> Democrat)
exec_records = []
for _, row in executive.iterrows():
    for term in row['terms']:
        if term['type'] == 'prez':
            party = "Democrat" if term.get('party') == 'Democratic' else term.get('party', 'Other')
            exec_records.append({
                'start': pd.to_datetime(term['start']), 
                # use current date for presidents still in office to allow time range math
                'end': pd.to_datetime(term.get('end')).replace(tzinfo=None) if term.get('end') else pd.Timestamp.today(), 
                'party': party
            })
df_pres = pd.DataFrame(exec_records)

# 100-year party lookup
pres_party_map = []
# create a lookup table that maps every month to the party in power at that time
for d in pd.date_range(start="1900-01-01", end=pd.Timestamp.today(), freq="MS"):
    active = df_pres[(df_pres['start'] <= d) & (df_pres['end'] > d)]
    if not active.empty:
        pres_party_map.append({'month': d.to_period('M'), 'appointer_party': active['party'].iloc[0]})
pres_df = pd.DataFrame(pres_party_map).set_index('month')

# Clean Dates & Merge
# replace day 00 with 01 in date strings to allow standard datetime conversion
positions['date_start'] = pd.to_datetime(positions['date_start'].astype(str).str.replace('-00', '-01'), errors='coerce')
positions['date_termination'] = pd.to_datetime(positions['date_termination'].astype(str).str.replace('-00', '-01'), errors='coerce').fillna(pd.Timestamp.today())

# force id columns to strings to ensure matching logic works during merge
positions['court_id'] = positions['court_id'].astype(str)
courts['id'] = courts['id'].astype(str)

# Merge
judges_df = positions.dropna(subset=['date_start']).merge(
    courts[['id', 'jurisdiction', 'full_name']], 
    left_on='court_id', right_on='id'
)

# Filter for Federal Courts (District & Appellate)
# District courts are the primary way to map 'by state'
judges_df = judges_df[judges_df['jurisdiction'].isin(['FD', 'F'])]

# Priority 1: Use the existing location_state column
# Priority 2: Extract from court name if P1 is missing
state_map = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
    'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}

def get_final_state(row):
    # Use existing abbreviation if valid
    if pd.notna(row['location_state']) and row['location_state'] in state_map.values():
        return row['location_state']
    # Fallback to name search
    # check for state names inside the full court title for missing state codes
    name = str(row['full_name'])
    for state, abbr in sorted(state_map.items(), key=lambda x: len(x[0]), reverse=True):
        if state in name: return abbr
    return None

judges_df['final_state'] = judges_df.apply(get_final_state, axis=1)
judges_df = judges_df.dropna(subset=['final_state'])

# Map Appointer Party
# determine the appointing party by joining judge start dates with the presidential lookup
judges_df['appoint_month'] = judges_df['date_start'].dt.to_period('M')
judges_df = judges_df.join(pres_df, on='appoint_month')

# Aggregate
records = []
years = range(1976, 2027)
for year in years:
    target_date = pd.Timestamp(f"{year}-01-01")
    # find all judges whose term covers the target snapshot date
    active = judges_df[(judges_df['date_start'] <= target_date) & (judges_df['date_termination'] > target_date)]
    # remove duplicate entries for individuals holding multiple positions in one year
    active = active.drop_duplicates(subset='person_id')
    # count active judges per state and per appointing party
    counts = active.groupby(['final_state', 'appointer_party']).size().unstack(fill_value=0)
    
    for s in sorted(state_map.values()):
        dem = counts.loc[s, 'Democrat'] if ('Democrat' in counts.columns and s in counts.index) else 0
        rep = counts.loc[s, 'Republican'] if ('Republican' in counts.columns and s in counts.index) else 0
        records.append({'year': year, 'state': s, 'democrat_judges': dem, 'republican_judges': rep})

final_df = pd.DataFrame(records)

print(final_df[final_df['state'] == 'CA'].head(10))

     year state  democrat_judges  republican_judges
4    1976    CA               22                 20
54   1977    CA               20                 25
104  1978    CA               20                 25
154  1979    CA               20                 25
204  1980    CA               22                 25
254  1981    CA               31                 23
304  1982    CA               30                 22
354  1983    CA               30                 28
404  1984    CA               28                 29
454  1985    CA               28                 34


# Visualization 2 Chart

This cell combines the geographic mapping of the federal judiciary with annual commodity price data. It uses `Altair` to link a choropleth map and a bar chart through a shared time-selection slider.

* **Annual Price Aggregation:** The code converts the high-resolution monthly commodity data into annual averages. This aligns the economic data with the judicial "snapshots" taken at the start of each year.
* **Judicial Lean Normalization:** It calculates a "Lean Score" (`judge_weight`) for every state. By subtracting Republican appointees from Democratic appointees and dividing by the total, it creates a scale from -1 (fully Republican-appointed) to +1 (fully Democratic-appointed).
* **State-to-Shape Mapping:** The script uses the standard ANSI numeric codes (`state_id`) to "stitch" the judicial statistics onto a TopoJSON map of the United States. It specifically uses the `albersUsa` projection to ensure Alaska and Hawaii are legibly positioned.
* **Coordinated Filtering:** A single `year_select` parameter is bound to a slider. Moving this slider simultaneously filters the map to show that year's judicial makeup and updates the bar chart to reflect that year's inflation-adjusted commodity prices.
* **Visual Stabilization:** To allow for accurate year-over-year comparisons, the code locks the bar chart's Y-axis to a fixed range based on the maximum price in the entire dataset. This prevents the "jumping" effect that occurs when axes auto-scale.

The final output is a responsive dashboard (`visualization2.json`) that allows users to explore how state-level judicial leanings and national commodity prices have shifted in tandem over the last five decades.

In [5]:
from vega_datasets import data

# numeric ids required for altair to match data rows to the us topojson shapes
state_id_map = {
    'AL': 1, 'AK': 2, 'AZ': 4, 'AR': 5, 'CA': 6, 'CO': 8, 'CT': 9, 'DE': 10, 'FL': 12, 'GA': 13, 
    'HI': 15, 'ID': 16, 'IL': 17, 'IN': 18, 'IA': 19, 'KS': 20, 'KY': 21, 'LA': 22, 'ME': 23, 
    'MD': 24, 'MA': 25, 'MI': 26, 'MN': 27, 'MS': 28, 'MO': 29, 'MT': 30, 'NE': 31, 'NV': 32, 
    'NH': 33, 'NJ': 34, 'NM': 35, 'NY': 36, 'NC': 37, 'ND': 38, 'OH': 39, 'OK': 40, 'OR': 41, 
    'PA': 42, 'RI': 44, 'SC': 45, 'SD': 46, 'TN': 47, 'TX': 48, 'UT': 49, 'VT': 50, 'VA': 51, 
    'WA': 53, 'WV': 54, 'WI': 55, 'WY': 56
}

# Aggregate adjusted_dataframes into annual averages for the dashboard
annual_commodities = []

for item, df in adjusted_dataframes.items():
    # Extract the year from the date column
    df['year'] = df['date'].dt.year
    
    # Calculate the average real price for each year starting from 1976
    annual_avg = df[df['year'] >= 1976].groupby('year')['real_price'].mean().reset_index()
    
    # Add the commodity name to label the data
    annual_avg['commodity'] = item
    annual_commodities.append(annual_avg)

# Combine all commodities into a single long-format DataFrame
commodity_df = pd.concat(annual_commodities, ignore_index=True)

# Ensure calculations are present
if 'judge_weight' not in final_df.columns:
    final_df['total_judges'] = final_df['democrat_judges'] + final_df['republican_judges']
    # normalize lean between negative one and positive one for color scale alignment
    final_df['judge_weight'] = (final_df['democrat_judges'] - final_df['republican_judges']) / final_df['total_judges']
    final_df['judge_weight'] = final_df['judge_weight'].fillna(0)

final_df['state_id'] = final_df['state'].map(state_id_map).fillna(0).astype(int)

# load external geographic shape data for us state boundaries
states_topo = alt.topo_feature(data.us_10m.url, 'states')

# create selection object for filtering all linked charts by the chosen year
year_slider = alt.binding_range(min=1976, max=2026, step=1, name='Select Year: ')
year_select = alt.selection_point(name='YearSel', fields=['year'], bind=year_slider, value=2026)

base_map = alt.Chart(final_df).mark_geoshape(
    stroke='white',
    strokeWidth=1
).transform_filter(
    # isolate data for the selected year before looking up shapes
    year_select
).transform_lookup(
    # attach geographic boundary data to the filtered judge statistics
    lookup='state_id',
    from_=alt.LookupData(states_topo, 'id'),
    as_='geo'
).encode(
    # specify that the looked up field contains geometry data
    shape=alt.Shape('geo:geojson'), 
    color=alt.Color('judge_weight:Q',
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    legend=alt.Legend(
                        title="Judicial Lean",
                        values=[-1, 0, 1],
                        # ternary logic to swap numeric values for text labels in legend
                        labelExpr="datum.value == -1 ? 'Republican' : (datum.value == 1 ? 'Democrat' : 'Neutral')"
                    )),
    tooltip=[
        alt.Tooltip('state:N', title='State'),
        alt.Tooltip('democrat_judges:Q', title='Dem Judges'),
        alt.Tooltip('republican_judges:Q', title='Rep Judges'),
        alt.Tooltip('judge_weight:Q', format='.2f', title='Lean Score')
    ]
).project(
    # use albers projection to automatically move alaska and hawaii below the mainland
    type='albersUsa'
).properties(
    width=600,
    height=400,
    title="US Judicial Lean by State (Appointing Party)"
)

# Get the absolute maximum price so the axis never rescales
max_price = commodity_df['real_price'].max()

bar_chart = alt.Chart(commodity_df).mark_bar().encode(
    x=alt.X('commodity:N', title=None, axis=alt.Axis(labelAngle=-45)),
    # lock y axis range to keep bars comparable as years change
    y=alt.Y('real_price:Q', title='Inflation-Adjusted Price ($)', scale=alt.Scale(domain=[0, max_price])),
    color=alt.Color('commodity:N', legend=None),
    tooltip=['commodity', alt.Tooltip('real_price:Q', format='$.2f')]
).transform_filter(
    year_select
).properties(
    width=250, 
    height=400,
    title="Commodity Prices"
)

# place map and bar chart side by side with shared selection parameter
dashboard = alt.hconcat(base_map, bar_chart).add_params(
    year_select
).resolve_scale(
    # ensure map lean colors and bar commodity colors do not merge
    color='independent'
)

dashboard.save('visualization2.json')
dashboard.display()

c:\Users\Nolan\anaconda3\envs\is445\Lib\site-packages\altair\vegalite\v6\api.py:3926: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  r = check._set_resolve(scale=core.ScaleResolveMap(*args, **kwargs))


alt.HConcatChart(...)